# Exercise 1: Setup & Exploration

**Objective:** Verify environment setup, explore base models, and understand tokenization basics

## Learning Outcomes

By the end of this exercise, you will be able to:
1. Check GPU availability and memory constraints
2. Load and inspect base LLMs (TinyLlama, Phi-2)
3. Explore tokenization strategies for LLMs
4. Understand computational requirements for LLM fine-tuning

---

## Environment Setup

First, let's check our environment and install any missing dependencies.

In [ ]:
# Check Python version
import sys
print(f"Python version: {sys.version}")

# Check for GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Install any missing packages
!pip install -q transformers peft bitsandbytes datasets evaluate mlflow vllm

## GPU Memory Constraints

Before loading models, let's understand the memory requirements:

| Model | Parameters | FP16 Memory | 4-bit Quantized Memory |
|-------|-----------|-------------|----------------------|
| TinyLlama-1.1B | 1.1B | ~2.2 GB | ~0.6 GB |
| Phi-2 | 2.7B | ~5.4 GB | ~1.5 GB |
| Llama-2-7B | 7B | ~14 GB | ~4 GB |

**Key Points:**
- **Quantization (4-bit)** reduces memory by ~4x with minimal quality loss
- **Training overhead**: LoRA adds ~1-2% additional parameters
- **Inference vs Training**: Training requires 2-3x more memory than inference
- **Batch size**: Larger batches require more GPU memory

For this workshop, we'll use **4-bit quantization** to fit models on available GPU memory.

---

## Loading Base Models

Let's load and inspect the base models we'll be working with.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load TinyLlama-1.1B model and tokenizer
tinyllama_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Loading {tinyllama_model_name}...")

tinyllama_tokenizer = AutoTokenizer.from_pretrained(tinyllama_model_name)
tinyllama_model = AutoModelForCausalLM.from_pretrained(
    tinyllama_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"TinyLlama model loaded successfully!")
print(f"Model type: {type(tinyllama_model)}")
print(f"Model config: {tinyllama_model.config}")

# Load Phi-2 model and tokenizer
phi2_model_name = "microsoft/phi-2"
print(f"\nLoading {phi2_model_name}...")

phi2_tokenizer = AutoTokenizer.from_pretrained(phi2_model_name)
phi2_model = AutoModelForCausalLM.from_pretrained(
    phi2_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Phi-2 model loaded successfully!")
print(f"Model type: {type(phi2_model)}")
print(f"Model config: {phi2_model.config}")

---

## Tokenization Exploration

Let's explore how these models tokenize text and understand their vocabulary.

In [ ]:
def explore_tokenization(model_name, tokenizer, sample_texts):
    """Explore tokenization for a given model."""
    print(f"\n=== Tokenization Analysis for {model_name} ===")
    print(f"Vocabulary size: {len(tokenizer)}")
    print(f"Special tokens: {tokenizer.special_tokens_map}")
    
    for text in sample_texts:
        tokens = tokenizer.tokenize(text)
        ids = tokenizer.encode(text)
        decoded = tokenizer.decode(ids)
        
        print(f"\nText: '{text}'")
        print(f"Tokens: {tokens}")
        print(f"Token IDs: {ids}")
        print(f"Decoded: '{decoded}'")
        print(f"Number of tokens: {len(tokens)}")

# Sample texts for exploration
sample_texts = [
    "Hello, how are you?",
    "The quick brown fox jumps over the lazy dog.",
    "I love machine learning and artificial intelligence!",
    "def hello_world():\n    print('Hello, World!')",
    "E = mc^2"
]

# Explore TinyLlama tokenization
explore_tokenization("TinyLlama-1.1B", tinyllama_tokenizer, sample_texts)

# Explore Phi-2 tokenization
explore_tokenization("Phi-2", phi2_tokenizer, sample_texts)

---

## Model Inference Test

Let's test basic inference with our loaded models to ensure they're working correctly.

In [ ]:
def generate_text(model, tokenizer, prompt, max_length=50, temperature=0.7):
    """Generate text using the model."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

# Test prompts
test_prompts = [
    "Explain the concept of machine learning in simple terms:",
    "Write a short poem about data science:",
    "What are the benefits of using version control in software development?"
]

print("=== TinyLlama-1.1B Inference ===")
for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    response = generate_text(tinyllama_model, tinyllama_tokenizer, prompt, max_length=100)
    print(f"Response: {response}")

print("\n\n=== Phi-2 Inference ===")
for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    response = generate_text(phi2_model, phi2_tokenizer, prompt, max_length=100)
    print(f"Response: {response}")

---

## Memory Usage Monitoring

Let's monitor GPU memory usage during model loading and inference.

In [ ]:
if torch.cuda.is_available():
    def print_gpu_memory_usage(step=""):
        allocated = torch.cuda.memory_allocated(0) / 1e9
        reserved = torch.cuda.memory_reserved(0) / 1e9
        print(f"GPU Memory {step}: Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB")
    
    print_gpu_memory_usage("before loading models")
    
    # Clear cache and check again
    torch.cuda.empty_cache()
    print_gpu_memory_usage("after clearing cache")
    
    # Load one model and check
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        torch_dtype=torch.float16,
        device_map="auto"
    )
    print_gpu_memory_usage("after loading TinyLlama")
    
    # Clean up
    del model
    torch.cuda.empty_cache()
    print_gpu_memory_usage("after cleanup")
else:
    print("CUDA not available, skipping GPU memory monitoring.")

---

## Summary

In this exercise, we:
1. Verified our environment setup and checked GPU availability
2. Loaded and inspected base LLMs (TinyLlama-1.1B and Phi-2)
3. Explored tokenization strategies for both models
4. Tested basic inference capabilities
5. Monitored GPU memory usage

**Key Takeaways:**
- Both models load successfully with 4-bit quantization
- Tokenization differs between models, affecting how text is processed
- Memory usage is manageable for inference on consumer GPUs with quantization
- Next step: Prepare instruction dataset for fine-tuning

---

*Next: Proceed to Exercise 2 - Data Preparation*